# GlitchGAN — GravitySpy Classification (Reference)

Reference classification notebook. Injection pipeline: fetch open data → `to_pycbc()` → whiten → pycbc copy → pycbc `+=` → gwpy TimeSeries.

**Set `path_to_repo` below to your local GravitySpy clone.**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import keras

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from glitchgan.tf import GlitchGAN
from glitchgan.tf.model_components import ArgmaxLayer, ReduceSumDotLayer
from glitchgan.utils import whitened_snr_scaling

#FIXME: your path to your Gravity Spy repo
path_to_repo = '/path/to/GravitySpy/'

# GlitchGAN model path (model is in the project models/ directory)
path_to_model = str(PROJECT_ROOT / 'models' / 'sidd-cqg-paper-O3-model.h5')

print('Project root  :', PROJECT_ROOT)
print('GravitySpy CNN:', path_to_model)
print('Model exists  :', Path(path_to_model).exists())

Project root  : /Users/tomdooney/Documents/Work/Projects/glitchgan
GravitySpy CNN: /Users/tomdooney/Documents/Work/Projects/glitchgan/models/sidd-cqg-paper-O3-model.h5
Model exists  : True


In [ ]:
# -----------------------------------------------
# Load GlitchGAN generator + generate signals
# -----------------------------------------------
LABEL_ORDER = [
    'Blip', 'Fast_Scattering', 'Koi_Fish',
    'Low_Frequency_Burst', 'Scattered_Light', 'Tomte', 'Whistle',
]
NUM_CLASSES      = len(LABEL_ORDER)
NOISE_DIM        = 100
NUM_PER_CLASS    = 100   # matches reference num_samples_per_class

generator_path = PROJECT_ROOT / 'weights' / 'tensorflow' / 'generator_210_keras3.keras'
gan = GlitchGAN()
gan.generator = keras.models.load_model(
    str(generator_path), compile=False,
    custom_objects={'ArgmaxLayer': ArgmaxLayer, 'ReduceSumDotLayer': ReduceSumDotLayer}
)
print(f'Loaded generator from: {generator_path.name}')

class_vectors     = np.repeat(np.eye(NUM_CLASSES, dtype='float32'), NUM_PER_CLASS, axis=0)
noise_vecs        = np.random.randn(NUM_CLASSES * NUM_PER_CLASS, NOISE_DIM).astype('float32')
generated_signals = gan.generator([noise_vecs, class_vectors], training=False).numpy()
labels            = np.repeat(np.array(LABEL_ORDER), NUM_PER_CLASS)

# aliases so the reference cell 45 code runs unchanged
X_bal    = generated_signals
y_labels = labels

print(f'generated_signals: {generated_signals.shape}')
print(f'labels           : {labels.shape}')

Loaded generator from: generator_210_keras3.keras
generated_signals: (700, 8192)
labels           : (700,)


### Gravity Spy classification

In [ ]:
import matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries
from matplotlib.ticker import ScalarFormatter
import pycbc
import sys
sys.path.insert(0, path_to_repo)
from gravityspy.classify import classify
from gravityspy.utils import utils
from gravityspy.plot import plot_qtransform
import random

ifo = 'H1'
# Define the GPS start and end times for the data segment you want to analyze
# 40s was my option but you can take something else
start, end = 1262540000, 1262540040 
init_time = -20 # the glitch is injected at t0=0, so the initial time of the series is -20
channel_name = f'{ifo}:GDS-CALIB_STRAIN' # to use later
srate = 4096
event_time = 0
init_time = -20 # the glitch is injected at t0=0, so the initial time of the series is -20

# Fetch open data from the specified interferometer (ifo)
noise = TimeSeries.fetch_open_data(ifo, start, end, sample_rate=srate)

# Convert the data from a gwpy TimeSeries object to a pycbc TimeSeries object
# This is necessary because we want to use pycbc-specific functions on the data
noise = noise.to_pycbc()


# - Whitening: This process flattens the power spectral density (PSD), making all frequencies equally weighted.
# Arguments:
# - len(noise) / (2 * srate): whitening segment length in seconds for estimating the noise PSD
# - len(noise) / (4 * srate): inverse spectrum truncation length (higher truncation reduces noise at high frequencies)
# - remove_corrupted=False: keeps all data points even if the whitening introduces edge effects
# - return_psd=True: also returns the estimated PSD along with the whitened noise
white_noise, psd = noise.whiten(len(noise) / (2 * srate),
                                len(noise) / (4 * srate),
                                remove_corrupted=False,
                                return_psd=True)

# Plotting white noise
plt.plot(white_noise[srate * 4 : -srate * 4])
plt.xlabel('Data points')
plt.ylabel('Amplitude')
plt.show()

length = noise.shape[-1]
len_glitch = X_bal[0].shape[-1]
t_inj = 0.5 * length / srate 
id_start = int((t_inj * srate / length) * len(white_noise)) - len_glitch // 2

# Take one sample

print(random.randint(0, len(X_bal)-1))
rand = random.randint(0, len(X_bal)-1)
sample = X_bal[rand]
lable = y_labels[rand]
print(lable) 


glitch = sample

snr = 50
glitch = whitened_snr_scaling(glitch, 20)

# Get the length (number of data points) of the noise array
length = noise.shape[-1]

# Get the length of the glitch signal (number of data points)
len_glitch = glitch.shape[-1]

# Define the injection time for the glitch (in seconds)
# This injects the glitch halfway through the noise data
t_inj = 0.5 * length / srate 

# Calculate the starting index where the glitch will be injected into white_noise
# Explanation:
# - (t_inj * srate / length): ratio to scale with respect to the noise length
# - len(white_noise): total number of points in white_noise
# - Subtract half the glitch length to center the glitch at t_inj
id_start = int((t_inj * srate / length) * len(white_noise)) - len_glitch // 2

# Add the glitch signal into the white_noise at the computed position
# This directly modifies white_noise by adding glitch amplitudes to the selected segment
white_noise[id_start:id_start + len_glitch] += glitch

# Plotting the injected noise
plt.plot(white_noise[id_start:id_start + len_glitch])
plt.xlabel('Data points')
plt.ylabel('Amplitude')
plt.show()

# We transform the numpy array to gwpy Timeseries
glitch_series = TimeSeries(white_noise, t0=init_time, sample_rate=srate,
                           name=ifo)


results_glitch = classify(event_time=0,
                          channel_name=channel_name,
                          path_to_cnn=path_to_model,
                          timeseries=glitch_series)
results_glitch['ml_label'].value[0], results_glitch['ml_confidence'].value[0]

/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gwpy/time/_ligotimegps.py:42: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1
INFO:panoptes_client:libmagic not operational, likely due to lack of shared libraries. Media MIME type determination will be based on file extensions.
INFO:gwpy.timeseries.timeseries:Found 1 possible sources
INFO:gwpy.timeseries.ti

564
Tomte


(np.str_('Tomte'), np.float32(0.99997056))

In [ ]:
import numpy as np
import random
from tqdm import tqdm
from gwpy.timeseries import TimeSeries
from gravityspy.classify import classify
import pycbc

# -----------------------------------------------
# CONFIGURATION
# -----------------------------------------------
num_samples_per_class = 100
snr_target = 50
ifo = 'H1'
srate = 4096
start, end = 1262540000, 1262540040
init_time = -20
channel_name = f'{ifo}:GDS-CALIB_STRAIN'

# -----------------------------------------------
# FETCH & WHITEN NOISE ONCE
# -----------------------------------------------
print("Fetching open data and whitening...")
noise = TimeSeries.fetch_open_data(ifo, start, end, sample_rate=srate)
noise = noise.to_pycbc()

white_noise, psd = noise.whiten(
    len(noise) / (2 * srate),
    len(noise) / (4 * srate),
    remove_corrupted=False,
    return_psd=True
)

# -----------------------------------------------
# STORAGE
# -----------------------------------------------
results = {
    'true_label': [],
    'pred_label': [],
    'confidence': []
}

label_order = ['Blip', 'Fast_Scattering', 'Koi_Fish',
               'Low_Frequency_Burst', 'Scattered_Light', 'Tomte', 'Whistle']

# -----------------------------------------------
# MAIN LOOP OVER CLASSES
# -----------------------------------------------
for class_label in tqdm(label_order, desc="Classifying generated glitches"):
    # Select 25 random samples from this class
    class_indices = np.where(labels == class_label)[0]
    chosen_indices = np.random.choice(class_indices, num_samples_per_class, replace=False)

    for idx in chosen_indices:
        glitch = generated_signals[idx].copy()

        # Scale to target SNR
        glitch = whitened_snr_scaling(glitch, snr_target)

        # Inject into white noise
        len_glitch = len(glitch)
        length = noise.shape[-1]
        t_inj = 0.5 * length / srate
        id_start = int((t_inj * srate / length) * len(white_noise)) - len_glitch // 2

        # Make a copy to avoid modifying the base noise
        injected_noise = white_noise.copy()
        injected_noise[id_start:id_start + len_glitch] += glitch

        # Convert to gwpy TimeSeries
        glitch_series = TimeSeries(injected_noise, t0=init_time, sample_rate=srate, name=ifo)

        # Classify using Gravity Spy
        try:
            result = classify(
                event_time=0,
                channel_name=channel_name,
                path_to_cnn=path_to_model,
                timeseries=glitch_series
            )
            pred_label = result['ml_label'].value[0]
            conf = result['ml_confidence'].value[0]
        except Exception as e:
            pred_label = "Error"
            conf = np.nan
            print(f"\u26a0\ufe0f Error classifying sample {idx}: {e}")

        # Store results
        results['true_label'].append(class_label)
        results['pred_label'].append(pred_label)
        results['confidence'].append(conf)

        print(class_label, pred_label, conf)

# -----------------------------------------------
# SAVE RESULTS
# -----------------------------------------------
import pandas as pd

df_results = pd.DataFrame(results)
df_results.to_csv("gravityspy_glitchgan_classification.csv", index=False)

print("\n\u2705 Classification complete!")
print(df_results.groupby(['true_label', 'pred_label']).size().unstack(fill_value=0))
print("\nSaved results to: gravityspy_glitchgan_classification.csv")

INFO:gwpy.timeseries.timeseries:Found 1 possible sources
INFO:gwpy.timeseries.timeseries:Attemping access with 'gwosc' [1/1]


Fetching open data and whitening...


Classifying generated glitches:   0%|          | 0/7 [00:00<?, ?it/s]/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given start smaller than current start, crop will begin when the Series actually starts.
  data = timeseries.crop(start_time, stop_time)
/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  data = timeseries.crop(start_time, stop_time)


Blip Blip 0.57568324


/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given start smaller than current start, crop will begin when the Series actually starts.
  data = timeseries.crop(start_time, stop_time)
/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  data = timeseries.crop(start_time, stop_time)


...

/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given start smaller than current start, crop will begin when the Series actually starts.
  data = timeseries.crop(start_time, stop_time)
/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  data = timeseries.crop(start_time, stop_time)


Koi_Fish Koi_Fish 0.7470665


/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given start smaller than current start, crop will begin when the Series actually starts.
  data = timeseries.crop(start_time, stop_time)
/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  data = timeseries.crop(start_time, stop_time)


Koi_Fish Blip 0.99998426


/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given start smaller than current start, crop will begin when the Series actually starts.
  data = timeseries.crop(start_time, stop_time)
/opt/homebrew/Caskroom/miniforge/base/envs/cdvgan/lib/python3.11/site-packages/gravityspy/utils/utils.py:108: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  data = timeseries.crop(start_time, stop_time)


In [ ]:
%matplotlib inline
df_results

In [ ]:
for lbl in label_order:
    sub = df_results[df_results.true_label == lbl]
    correct = (sub.pred_label == lbl).sum()
    print(f"{lbl}: {correct}/{len(sub)} ({100*correct/len(sub):.0f}%)")